# 练习版：基于AReaL强化学习训练长程搜索智能体（Search Agent）

本文件由 `search_agent_zh.ipynb` 复制而来，用作练习。

本教程利用AReaL中的基本组件快速搭建一个强化学习流程，用来训练一个可以进行长程搜索的智能体。

该教程包括以下步骤：
1. 实验准备（包括从yaml加载实验配置，配置环境变量，启动SGLang服务器，启动本地RAG服务器，加载训练数据集）
2. 定义简单的工作流，多次调用搜索工具；
3. 每次生成**多条**轨迹（i.e., GRPO)；
4. 测试工作流；
5. 将工作流接入端到端GRPO强化学习训练；

## 实验准备
### 加载实验配置

通过`load_expr_config`加载预定义的asearcher基于local RAG训练的yaml实验配置模板。

实验配置模板内配置了优化器、模板、学习率等参数，可以直接使用。

In [1]:
from dataclasses import dataclass, field

from areal.api.cli_args import GRPOConfig, load_expr_config

import os
os.chdir("/home/ubuntu/AReaL")
print(os.getcwd())

@dataclass
class AgentRLConfig(GRPOConfig):
    max_turns: int = field(
        default=32, metadata={"help": "maximum number of turns per trajectory"}
    )


args = ["--config", "examples/search_agent/local_0.5b_single_gpu.yaml"]
config, _ = load_expr_config(args, AgentRLConfig)
config: AgentRLConfig

/home/ubuntu/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-05-24 22:44:53,444	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


/home/ubuntu/AReaL


### 配置环境变量

我们预先分配SGLang服务器和PyTorch分布式启动的IP地址和端口，并设置相应的环境变量。

这些环境变量会在引擎初始化时被读取。

***在非notebook环境下，这些环境变量会被launcher设置，用户无需自行设置。***

In [2]:
import os
import subprocess
import sys

os.chdir("/home/ubuntu/AReaL")

SGLANG_PORT, MASTER_PORT = 11451, 14514
SGLANG_HOST = "127.0.0.1"

os.environ["CUDA_HOME"] = "/usr/local/cuda"
os.environ["PATH"] = f"{os.environ['CUDA_HOME']}/bin:" + os.environ.get("PATH", "")
os.environ["LD_LIBRARY_PATH"] = f"{os.environ['CUDA_HOME']}/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["CC"] = "/usr/bin/gcc-12"
os.environ["CXX"] = "/usr/bin/g++-12"
os.environ["CUDAHOSTCXX"] = "/usr/bin/g++-12"

# ----------------------------------------------------------------------------
# Environment variables used by inference/train engines
os.environ["AREAL_LLM_SERVER_ADDRS"] = f"{SGLANG_HOST}:{SGLANG_PORT}"
os.environ["MASTER_ADDR"] = "127.0.0.1"
os.environ["MASTER_PORT"] = str(MASTER_PORT)
os.environ["RANK"] = "0"
os.environ["WORLD_SIZE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "true"
os.environ["LOCAL_RANK"] = "0"

### 启动SGLang服务器

AReaL默认采用训推分离式架构，推理和训练异步执行，能够打满GPU资源、快速完成端到端训练。

在这个练习样例中，强化学习的算法编排（GRPO）和SGLang推理服务器共用GPU 0。

下面的代码块在GPU 0上启动对应的推理服务。

本次教程中使用`Qwen/Qwen2.5-0.5B-Instruct`作为例子。

In [3]:
# 启动sglang server
from areal.api.cli_args import SGLangConfig

config.sglang.log_level = "info"
config.sglang.decode_log_interval = 10
sglang_cmd = SGLangConfig.build_cmd(
    config.sglang,
    tp_size=1,
    base_gpu_id=0,
    host=SGLANG_HOST,
    port=SGLANG_PORT,
)
sglang_process = subprocess.Popen(
    sglang_cmd,
    stdout=sys.stdout,
    stderr=sys.stderr,
)

print("sglang process is launched")

sglang process is launched


### 加载训练数据集

使用HuggingFace `datasets` 包加载训练数据集，并查看数据集格式

In [4]:
# load search dataset
from datasets import load_dataset

print(f"dataset is at {config.train_dataset.path}")
dataset = load_dataset(
    path="json",
    split="train",
    data_files=config.train_dataset.path,
)
print(f">>> dataset column names: {dataset.column_names}")
print(f">>> example data: {dataset[0]}")

dataset is at examples/search_agent/practice_tiny.jsonl
>>> dataset column names: ['question', 'answer']
>>> example data: {'question': 'What is the capital city of China?', 'answer': ['Beijing']}


### 导入必要的包和模块


In [5]:
import asyncio
import json
import os
import sys
import time
import uuid

import torch
from datasets import load_dataset
from transformers import AutoTokenizer

from areal.api.alloc_mode import ModelAllocation
from areal.api.cli_args import (
    GenerationHyperparameters,
    load_expr_config,
)
from areal.api.engine_api import InferenceEngine
from areal.api.io_struct import (
    FinetuneSpec,
    ModelRequest,
    WeightUpdateMeta,
)
from areal.engine.fsdp_engine import FSDPPPOActor
from areal.engine.sglang_remote import RemoteSGLangEngine
from areal.utils.data import concat_padded_tensors, tensor_container_to

tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_path)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

/home/ubuntu/AReaL/areal/engine/fsdp_utils/grad.py:37: UserWarning: Transformer Engine and Apex are not installed. Falling back to local implementations of multi_tensor_applier, multi_tensor_l2norm, and multi_tensor_scale
  warnings.warn(


(AReaL) 20260524-22:45:00.467 TreeAttentionFSDP INFO: Compiled torch flex attention. Options: {'epilogue_fusion': True, 'max_autotune': False, 'shape_padding': True, 'trace.enabled': False, 'triton.cudagraphs': False}, dynamic: True
(AReaL) 20260524-22:45:00.469 TreeAttentionFSDP INFO: Using block mask in flex attention, block size: 128


/home/ubuntu/.venv/lib/python3.12/site-packages/megatron/core/distributed/fsdp/src/megatron_fsdp/mixed_precision.py:115: UserWarning: Transformer Engine and Apex are not installed. Falling back to local implementations of multi_tensor_applier and multi_tensor_scale
  warnings.warn(
/home/ubuntu/.venv/lib/python3.12/site-packages/megatron/core/optimizer/__init__.py:24: UserWarning: Transformer Engine and Apex are not installed. Falling back to Torch optimizers.
  warnings.warn(
/home/ubuntu/.venv/lib/python3.12/site-packages/megatron/core/optimizer/optimizer.py:29: UserWarning: Transformer Engine and Apex are not installed. Falling back to local implementations of multi_tensor_applier and multi_tensor_scale
  warnings.warn(
/home/ubuntu/.venv/lib/python3.12/site-packages/megatron/core/optimizer/clip_grads.py:32: UserWarning: Transformer Engine and Apex are not installed. Falling back to local implementations of multi_tensor_applier, multi_tensor_l2norm, and multi_tensor_scale
  warnings

使用`torchdata.stateful_dataloader.StatefulDataLoader` 作为dataloader

In [6]:
# setup dataloader

from itertools import cycle

from torchdata.stateful_dataloader import StatefulDataLoader

dataloader = StatefulDataLoader(
    dataset,
    batch_size=config.train_dataset.batch_size,
    shuffle=True,
    collate_fn=lambda x: x,
    drop_last=True,
)

data_generator = cycle(dataloader)

ft_spec = FinetuneSpec(
    total_train_epochs=config.total_train_epochs,
    dataset_size=len(dataloader) * config.train_dataset.batch_size,
    train_batch_size=config.train_dataset.batch_size,
)

batch = next(data_generator)
print(f">>> The type of a batch is: {type(batch)}\n")
print(f">>> Each piece of data has keys: {batch[0].keys()}\n")
print(f">>> Example input question: {batch[0]['question']}\n")

>>> The type of a batch is: <class 'list'>

>>> Each piece of data has keys: dict_keys(['question', 'answer'])

>>> Example input question: What is the largest ocean on Earth?



### 配置搜索工具

本地RAG服务器部署方式请见：[ASearcher 仓库](https://github.com/inclusionAI/ASearcher/blob/main/docs/training.md#b-training-a-search-agent-with-local-knowledge-base) - Step 2.

通过5001端口给本地RAG服务器发送查询，并接收结果。

如下展示了一个搜索关键词"China"的例子。

In [7]:
# setup tool

import aiohttp

TOOL_SERVER_ADDR = "localhost:5001"

FALLBACK_SEARCH_CORPUS = [
    {
        "wikipedia_title": "Beijing",
        "contents": "Beijing is the capital city of China.",
    },
    {
        "wikipedia_title": "William Shakespeare",
        "contents": "William Shakespeare wrote the tragedy Hamlet.",
    },
    {
        "wikipedia_title": "Mars",
        "contents": "Mars is often called the Red Planet.",
    },
    {
        "wikipedia_title": "Pacific Ocean",
        "contents": "The Pacific Ocean is the largest ocean on Earth.",
    },
]


def fallback_search(queries, topk=5):
    results = []
    for query in queries:
        query = query or ""
        query_tokens = set(str(query).lower().split())
        scored = []
        for item in FALLBACK_SEARCH_CORPUS:
            text = f"{item['wikipedia_title']} {item['contents']}".lower()
            score = sum(token in text for token in query_tokens)
            scored.append((score, item))
        scored.sort(key=lambda x: x[0], reverse=True)
        results.append([item for _, item in scored[:topk]])
    return results


async def call_search_tool(**req_meta):
    req_meta["queries"] = [q for q in req_meta.get("queries", []) if q]
    if not req_meta["queries"]:
        return [[]]

    try:
        async with aiohttp.ClientSession() as session:
            async with session.post(
                f"http://{TOOL_SERVER_ADDR}/retrieve",
                json=req_meta,
                timeout=aiohttp.ClientTimeout(total=120, sock_connect=3),
            ) as response:
                response.raise_for_status()
                res = await response.json()
                return res["result"]
    except aiohttp.ClientError as exc:
        print(f"[search fallback] local RAG server is unavailable: {exc}")
        return fallback_search(req_meta["queries"], topk=req_meta.get("topk", 5))


result = (await call_search_tool(queries=["China"], topk=5, return_scores=False))[0]
print(json.dumps(result, indent=4))

[
    {
        "wikipedia_id": "local-1",
        "wikipedia_title": "Beijing",
        "url": "https://en.wikipedia.org/w/index.php?title=Beijing",
        "contents": "Beijing\nBeijing is the capital city of China. It is one of the world's major political and cultural centers.",
        "id": 0
    },
    {
        "wikipedia_id": "573",
        "wikipedia_title": "Alchemy",
        "url": "https://en.wikipedia.org/w/index.php?title=Alchemy",
        "contents": "Alchemy\ncentered in China and its zone of cultural influence; Indian alchemy, centered on the Indian subcontinent; and Western alchemy, which occurred around the Mediterranean and whose center has shifted over the millennia from Greco-Roman Egypt, to the Islamic world, and finally medieval Europe. Chinese alchemy was closely connected to Taoism and Indian alchemy with the Dharmic faiths, whereas Western alchemy developed its own philosophical system that was largely independent of, but influenced by, various Western religi

## 定义简单的智能体工作流

### 模型输出

使用prompt控制模型输出，模型输出应遵循特定格式：
- `<think></think>` 包含模型思考过程
- `<search></search>` 包含给本地RAG服务器的查询
- `<answer></answer>` 包含模型输出的答案

此外，使用`<information></information>` 包含RAG服务器返回的查询内容。

In [8]:
PROMPT_TEMPLATE = """A conversation between User and Assistant. The user asks a question, and the Assistant answers it. The Assistant analyzes the given question and information in the mind, retains important relevant information, calls a search engine to find necessary information, accesses web pages with certain urls, and provides the user with the answer. The Assistant conducts search by <search> query </search> and the top search results will be returned between <information> and </information>. The reasoning processes are enclosed within <think> </think>. Finally, the Assistant provides answer inside <answer> and </answer>, i.e. <answer> answer here </answer>. If there are multiple queries, ensure all answers are enclosed within <answer> </answer>, seperated with comma.

User:
{question}

Assistant:
<think>"""

batch = next(data_generator)
prompt = PROMPT_TEMPLATE.format(question=batch[0]["question"])

print(f">>> PROMPT: {prompt}")

>>> PROMPT: A conversation between User and Assistant. The user asks a question, and the Assistant answers it. The Assistant analyzes the given question and information in the mind, retains important relevant information, calls a search engine to find necessary information, accesses web pages with certain urls, and provides the user with the answer. The Assistant conducts search by <search> query </search> and the top search results will be returned between <information> and </information>. The reasoning processes are enclosed within <think> </think>. Finally, the Assistant provides answer inside <answer> and </answer>, i.e. <answer> answer here </answer>. If there are multiple queries, ensure all answers are enclosed within <answer> </answer>, seperated with comma.

User:
Which planet is known as the Red Planet?

Assistant:
<think>


通过`RemoteSGlangEngine`向已经启动的SGLang服务器发送生成请求，测试上述prompt控制下的模型输出。

In [9]:
asyncio.get_running_loop()

<_UnixSelectorEventLoop running=True closed=False debug=False>

In [10]:
# initialize inference engine
rollout_engine = RemoteSGLangEngine(config.rollout)
rollout_engine.initialize()

# generation config
gconfig = GenerationHyperparameters(
    max_new_tokens=512, stop=["</search>", "</answer>", "</access>"]
)

# tokenize the prompt
input_ids = tokenizer([prompt], add_special_tokens=False)["input_ids"][0]
req = ModelRequest(rid=uuid.uuid4().hex, input_ids=input_ids, gconfig=gconfig)

# generate rollout with inference engine
resp = await rollout_engine.agenerate(req)
completion_str = tokenizer.decode(resp.output_tokens)

# logging
print(f">>> prompt str: {tokenizer.decode(resp.input_tokens)}")
print(f">>> generated: {tokenizer.decode(resp.output_tokens)}")

(AReaL) 20260524-22:45:03.880 [RemoteInfEngine Rank 6274ef5198614f5c9179128a319afe47] INFO: Failed to get server addresses from name_resolve, falling back to environment variable.
(AReaL) 20260524-22:45:03.882 [RemoteInfEngine Rank 6274ef5198614f5c9179128a319afe47] INFO: Get server addresses from environment variable.
(AReaL) 20260524-22:45:03.882 [RemoteInfEngine Rank 6274ef5198614f5c9179128a319afe47] INFO: Waiting for server ready...
(AReaL) 20260524-22:45:28.923 [RemoteInfEngine Rank 6274ef5198614f5c9179128a319afe47] INFO: Servers are all ready!
>>> prompt str: A conversation between User and Assistant. The user asks a question, and the Assistant answers it. The Assistant analyzes the given question and information in the mind, retains important relevant information, calls a search engine to find necessary information, accesses web pages with certain urls, and provides the user with the answer. The Assistant conducts search by <search> query </search> and the top search results will

#### 解析智能体工具调用

定义`parse_search_query`函数从模型输出解析调用搜索工具的查询。

In [11]:
# TODO(练习 1): 实现工具调用解析函数。

import re


def parse_search_query(text):
    pattern = r"<search>(.*?)</search>"
    match = re.search(pattern, text, re.DOTALL)
    if match:
        return match.group(1).strip()
    else:
        return None


test_tool_str = "<think> I would like to search for AI.</think>\n<search> Artificial Intelligence </search>"
print(">>> input: ", test_tool_str)
print(">>> search query: ", parse_search_query(test_tool_str))

>>> input:  <think> I would like to search for AI.</think>
<search> Artificial Intelligence </search>
>>> search query:  Artificial Intelligence


在模型输出上测试工具调用解析函数`parse_search_query`。

In [12]:
# generate rollout with inference engine
resp = await rollout_engine.agenerate(req)
completion_str = tokenizer.decode(resp.output_tokens)

# logging
print(f">>> prompt str: {tokenizer.decode(resp.input_tokens)}")
print(f">>> generated: {tokenizer.decode(resp.output_tokens)}")
print(f">>> search query: {parse_search_query(completion_str)}")

>>> prompt str: A conversation between User and Assistant. The user asks a question, and the Assistant answers it. The Assistant analyzes the given question and information in the mind, retains important relevant information, calls a search engine to find necessary information, accesses web pages with certain urls, and provides the user with the answer. The Assistant conducts search by <search> query </search> and the top search results will be returned between <information> and </information>. The reasoning processes are enclosed within <think> </think>. Finally, the Assistant provides answer inside <answer> and </answer>, i.e. <answer> answer here </answer>. If there are multiple queries, ensure all answers are enclosed within <answer> </answer>, seperated with comma.

User:
Which planet is known as the Red Planet?

Assistant:
<think>
>>> generated: Typically, Uranus is known as the Red Planet as it is so named due to its characteristic coloration, primarily composed mainly of methan

#### 解析智能体答案

定义 `parse_answer` 函数从模型输出中解析模型答案。

In [13]:
# TODO(练习 2): 实现答案解析函数。


def parse_answer(text):
    pattern = r"<answer>(.*?)</answer>"
    match = re.search(pattern, text, re.DOTALL)
    if match:
        return match.group(1).strip()
    else:
        return None


test_answer_str = (
    "<think> I already found the answer! </think>\n<answer> 1997 </answer>"
)
print(">>> input: ", test_answer_str)
print(">>> answer: ", parse_answer(test_answer_str))

>>> input:  <think> I already found the answer! </think>
<answer> 1997 </answer>
>>> answer:  1997


在模型输出上测试答案解析函数`parse_answer`。

In [14]:
# generate rollout with inference engine
resp = await rollout_engine.agenerate(req)
completion_str = tokenizer.decode(resp.output_tokens)

# logging
print(f">>> prompt str: {tokenizer.decode(resp.input_tokens)}")
print(f">>> generated: {tokenizer.decode(resp.output_tokens)}")
print(f">>> answer: {parse_answer(completion_str)}")

>>> prompt str: A conversation between User and Assistant. The user asks a question, and the Assistant answers it. The Assistant analyzes the given question and information in the mind, retains important relevant information, calls a search engine to find necessary information, accesses web pages with certain urls, and provides the user with the answer. The Assistant conducts search by <search> query </search> and the top search results will be returned between <information> and </information>. The reasoning processes are enclosed within <think> </think>. Finally, the Assistant provides answer inside <answer> and </answer>, i.e. <answer> answer here </answer>. If there are multiple queries, ensure all answers are enclosed within <answer> </answer>, seperated with comma.

User:
Which planet is known as the Red Planet?

Assistant:
<think>
>>> generated:  To answer this question, I first need to recall the classification of the planet Mars. </think> 
<think> Mars is classified as a planet

### 奖励函数

我们默认使用F1 score作为奖励函数。

In [15]:
# TODO(练习 3): 实现简化版 F1 reward。


def f1_score(pred_ans, gt):
    if isinstance(gt, list):
        return max(f1_score(pred_ans, x) for x in gt)

    pred_tokens = set(pred_ans.lower().split())
    gt_tokens = set(gt.lower().split())

    if not pred_tokens and not gt_tokens:
        return 1.0
    if not pred_tokens or not gt_tokens:
        return 0.0

    common_tokens = pred_tokens & gt_tokens
    precision = len(common_tokens) / len(pred_tokens)
    recall = len(common_tokens) / len(gt_tokens)

    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


print(
    "f1_score('James Bond', 'James Bond'): {:.2f}".format(
        f1_score("James Bond", "James Bond")
    )
)
print(
    "f1_score('James Smith', 'James Bond'): {:.2f}".format(
        f1_score("James Smith", "James Bond")
    )
)

f1_score('James Bond', 'James Bond'): 1.00
f1_score('James Smith', 'James Bond'): 0.50


### 定义搜索智能体工作流

实现搜索智能体 (Search Agent) 的工作流非常简单，从一个初始问题出发，在每一轮：
1. 调用推理引擎进行生成，当生成到EOS、`</search>`、`</answer>`之一时停止生成
2. 如果检测到搜索查询，调用搜索工具，并将搜索结果加入到历史中
3. 如果检测到答案，计算奖励并退出循环

最后将数据组合成训练需要的形式

In [16]:
# TODO(练习 4): 实现 search result 格式化和 Search Agent rollout 工作流。


def format_search_results(search_results):
    return (
        "\n\n<information>\n"
        + "\n\n".join(
            [
                '<p title="{}">\n{}\n</p>'.format(
                    r["wikipedia_title"],
                    r["contents"],
                )
                for r in search_results
            ]
        )
        + "\n</information>"
    )


class SearchAgentWorkflow:
    def __init__(self, gconfig, tokenizer, max_tokens, max_turns, verbose):
        self.gconfig = gconfig
        self.tokenizer = tokenizer
        self.max_tokens = max_tokens
        self.max_turns = max_turns
        self.verbose = verbose

    async def arun_episode(self, engine: InferenceEngine, data):
        prompt = PROMPT_TEMPLATE.format(question=data["question"])
        rid = uuid.uuid4().hex

        input_ids = self.tokenizer.encode(prompt, add_special_tokens=False)
        logprobs = [0.0] * len(input_ids)
        loss_mask = [0] * len(input_ids)

        answer, reward = None, 0
        num_turns = 0

        while num_turns < self.max_turns and len(input_ids) < self.max_tokens:
            num_turns += 1

            # 1. 调用推理引擎生成一段模型输出。
            req = ModelRequest(rid=rid, input_ids=input_ids, gconfig=self.gconfig)
            resp = await engine.agenerate(req)
            # 2. 将模型输出加入 trajectory，并更新 logprobs / loss_mask。
            logprobs.extend(resp.output_logprobs)
            loss_mask.extend([1] * len(resp.output_tokens))
            input_ids.extend(resp.output_tokens)
            # 3. 如果模型发起 <search>，调用搜索工具，并把 <information> 加回上下文。
            search_query = parse_search_query(self.tokenizer.decode(resp.output_tokens))
            if search_query:
                search_results = (await call_search_tool(
                    queries=[search_query], topk=1, return_scores=False
                ))[0]
                formatted_results = format_search_results(search_results)
                search_token_ids = self.tokenizer.encode(
                    formatted_results,
                    add_special_tokens=False,
                )

                input_ids.extend(search_token_ids)
                logprobs.extend([0.0] * len(search_token_ids))
                loss_mask.extend([0] * len(search_token_ids))
            # 4. 如果模型给出 <answer>，计算 reward 并结束。
            answer =parse_answer(self.tokenizer.decode(resp.output_tokens))
            if answer:
                reward = max(f1_score(answer, gt) for gt in data["answer"])
                break

        if self.verbose:
            print(f"[LOGGING] turns={num_turns} length={len(input_ids)} reward={reward}")

        res = dict(
            input_ids=torch.tensor(input_ids),
            logprobs=torch.tensor(logprobs),
            loss_mask=torch.tensor(loss_mask),
            rewards=torch.tensor(float(reward)),
            attention_mask=torch.ones(len(input_ids), dtype=torch.bool),
        )
        res = {k: v.unsqueeze(0) for k, v in res.items()}
        return res

#### 测试搜索智能体工作流

1. 创建推理引擎；
2. 创建工作流，设定`max_new_tokens`, `max_turns` 和 `max_tokens`；
3. 将工作流传入推理引擎进行批量生成。

In [17]:
# initialize inference engine
rollout = RemoteSGLangEngine(config.rollout)
rollout.initialize()

try:
    # TODO(练习 5): 创建 SearchAgentWorkflow，并测试多条样本的 rollout。
    workflow = SearchAgentWorkflow(
        gconfig=GenerationHyperparameters(
        max_new_tokens=256,
        stop=["</answer>", "</search>"],
        ),
        tokenizer=tokenizer,
        max_tokens=2048,
        max_turns=5,
        verbose=True,
    )
    sample_data = next(data_generator)[:2]
    res = await asyncio.gather(
        *[workflow.arun_episode(rollout, sample_data[i]) for i in range(2)]
    )
    res = concat_padded_tensors(res)
    print(res)

    # log the trajectories
    traj_lens = res["attention_mask"].sum(dim=1).numpy().tolist()
    for i in range(2):
        token_ids = res["input_ids"][i, : traj_lens[i]]
        print(f">>> Trajectory {i} >>>\n{tokenizer.decode(token_ids)}")
finally:
    rollout.destroy()

(AReaL) 20260524-22:45:33.298 [RemoteInfEngine Rank 3034bdce86464e18875b98e67d8854b7] INFO: Failed to get server addresses from name_resolve, falling back to environment variable.
(AReaL) 20260524-22:45:33.299 [RemoteInfEngine Rank 3034bdce86464e18875b98e67d8854b7] INFO: Get server addresses from environment variable.
(AReaL) 20260524-22:45:33.299 [RemoteInfEngine Rank 3034bdce86464e18875b98e67d8854b7] INFO: Waiting for server ready...
(AReaL) 20260524-22:45:34.303 [RemoteInfEngine Rank 3034bdce86464e18875b98e67d8854b7] INFO: Servers are all ready!
[LOGGING] turns=2 length=365 reward=0.6666666666666666
[LOGGING] turns=5 length=1443 reward=0
{'input_ids': tensor([[   32, 10435,  1948,  ...,     0,     0,     0],
        [   32, 10435,  1948,  ...,   323,   279,  3853]]), 'logprobs': tensor([[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [ 0.0000,  0.0000,  0.0000,  ..., -2.1222, -2.4680, -5.2443]]), 'loss_mask': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0, 

### 让智能体工作流对每个问题生成多条轨迹


类似GRPO的算法需要针对每个问题生成一组多条轨迹。

我们可以通过一个asyncio的并行技巧同时高效地生成多个轨迹。

In [18]:
# TODO(练习 6): 实现 group generation for GRPO。


from areal.api.workflow_api import RolloutWorkflow


class GroupedSearchAgentWorkflow(RolloutWorkflow):
    def __init__(self, gconfig, tokenizer, max_tokens, max_turns, group_size, verbose):
        self.gconfig = gconfig
        self.tokenizer = tokenizer
        self.max_tokens = max_tokens
        self.max_turns = max_turns
        self.group_size = group_size
        self.verbose = verbose

    async def arun_episode(self, engine, data):
        workflows = [SearchAgentWorkflow(
                        gconfig=self.gconfig.new(n_samples=1),
                        tokenizer=self.tokenizer,
                        max_tokens=self.max_tokens,
                        max_turns=self.max_turns,
                        verbose=self.verbose,
                        ) for _ in range(self.group_size)]

        tasks = [workflow.arun_episode(engine, data) for workflow in workflows]
        results = await asyncio.gather(*tasks)
        results = concat_padded_tensors(results)
        return results

In [19]:
# initialize inference engine
rollout = RemoteSGLangEngine(config.rollout)
rollout.initialize()
try:
    # TODO(练习 7): 创建 GroupedSearchAgentWorkflow，并测试 group rollout。
    workflow = GroupedSearchAgentWorkflow(
        gconfig=GenerationHyperparameters(
        max_new_tokens=256,
        stop=["</answer>", "</search>"],
        ),
        tokenizer=tokenizer,
        max_tokens=2048,
        max_turns=5,
        group_size=4,
        verbose=True,
    )
    sample_data = next(data_generator)[:2]
    res = rollout.rollout_batch(sample_data, workflow=workflow)
    print(res)
finally:
    rollout.destroy()

(AReaL) 20260524-22:45:45.306 [RemoteInfEngine Rank ae90a700c9c44b9a933e6a80ecb214df] INFO: Failed to get server addresses from name_resolve, falling back to environment variable.
(AReaL) 20260524-22:45:45.307 [RemoteInfEngine Rank ae90a700c9c44b9a933e6a80ecb214df] INFO: Get server addresses from environment variable.
(AReaL) 20260524-22:45:45.307 [RemoteInfEngine Rank ae90a700c9c44b9a933e6a80ecb214df] INFO: Waiting for server ready...
(AReaL) 20260524-22:45:46.311 [RemoteInfEngine Rank ae90a700c9c44b9a933e6a80ecb214df] INFO: Servers are all ready!
[LOGGING] turns=2 length=251 reward=0.5
[LOGGING] turns=1 length=226 reward=0.0
[LOGGING] turns=1 length=240 reward=1.0
[LOGGING] turns=1 length=292 reward=0.0
[LOGGING] turns=5 length=452 reward=0
[LOGGING] turns=5 length=752 reward=0
(AReaL) 20260524-22:45:53.063 [RemoteInfEngine Rank ae90a700c9c44b9a933e6a80ecb214df] WARNING: Trajectory missing 'versions' field, defaulting to current inference engine version.
[LOGGING] turns=5 length=831 

## 将智能体工作流接入强化学习训练流程

上面我们已经测试好了负责的推理工作流，接下来我们需要将这个工作流接入到训练过程中。

这需要我们额外创建一个专门针对PPO的训练引擎，并在training loop中循环调用推理和训练。

In [20]:
config.actor.log_agent_stats = False


In [ ]:
# 100-step training with wandB logging

import wandb
from areal.utils import stats_tracker

config.actor.log_agent_stats = False

train_steps = 100
warmup_steps = 1
wandb_project = "asearcher"
run_name = f"search-0.5b-single-gpu-tiny-rag-{train_steps}steps"

wandb_run = wandb.init(
    project=wandb_project,
    name=run_name,
    config={
        "train_steps": train_steps,
        "actor_path": config.actor.path,
        "rollout_backend": config.rollout.backend,
        "actor_backend": config.actor.backend,
        "group_size": 4,
        "batch_size": config.train_dataset.batch_size,
        "max_turns": 5,
        "max_tokens": 2048,
        "max_new_tokens": 256,
        "dataset_path": config.train_dataset.path,
        "rag_server": TOOL_SERVER_ADDR,
        "weight_update_mode": config.actor.weight_update_mode,
    },
)


def reset_areal_stats_tracker():
    from collections import defaultdict

    for tracker in stats_tracker.TRACKERS.values():
        tracker.denominators = {}
        tracker.reduce_types = {}
        tracker.stats = defaultdict(list)


def clear_stale_disk_weight_entries():
    import shutil
    from pathlib import Path

    root = Path(config.cluster.name_resolve.nfs_record_root)
    update_dir = root / "ubuntu" / config.experiment_name / config.trial_name / "update_weights_from_disk"
    if update_dir.exists():
        shutil.rmtree(update_dir)
        print(f"cleared stale disk weight entries: {update_dir}")


def rollout_metrics(batch):
    rewards = torch.cat([traj["rewards"].detach().float().cpu().reshape(-1) for traj in batch])
    seq_lens = torch.cat([
        traj["attention_mask"].detach().cpu().sum(dim=-1).float().reshape(-1)
        for traj in batch
    ])
    response_lens = torch.cat([
        traj["loss_mask"].detach().cpu().sum(dim=-1).float().reshape(-1)
        for traj in batch
    ])
    return {
        "rollout/reward_mean": rewards.mean().item(),
        "rollout/reward_max": rewards.max().item(),
        "rollout/reward_min": rewards.min().item(),
        "rollout/success_rate": (rewards > 0).float().mean().item(),
        "rollout/seq_len_mean": seq_lens.mean().item(),
        "rollout/response_len_mean": response_lens.mean().item(),
        "rollout/n_trajs": int(rewards.numel()),
    }


workflow = GroupedSearchAgentWorkflow(
    gconfig=GenerationHyperparameters(
        max_new_tokens=256,
        stop=["</answer>", "</search>"],
    ),
    tokenizer=tokenizer,
    max_tokens=2048,
    max_turns=5,
    group_size=4,
    verbose=False,
)

rollout_alloc = ModelAllocation.from_str("sglang:d1p1t1")
actor_alloc = ModelAllocation.from_str("fsdp:d1p1t1")
parallel_strategy = actor_alloc.parallel

reset_areal_stats_tracker()
clear_stale_disk_weight_entries()

actor = FSDPPPOActor(config=config.actor)
actor.create_process_group(parallel_strategy=parallel_strategy)
actor.initialize(None, ft_spec)

rollout = RemoteSGLangEngine(config.rollout)
rollout.initialize(train_data_parallel_size=parallel_strategy.dp_size)

weight_update_meta = WeightUpdateMeta.from_disk(
    experiment_name=config.experiment_name,
    trial_name=config.trial_name,
    file_root=config.cluster.fileroot,
    name="default",
    clear_checkpoint_after_load=True,
)
actor.connect_engine(rollout, weight_update_meta)

times = []
for global_step in range(train_steps):
    tik = time.perf_counter()

    batch = actor.rollout_batch(next(data_generator), workflow=workflow)
    metrics = rollout_metrics(batch)

    batch = tensor_container_to(batch, actor.device)

    logps = actor.compute_logp(batch)
    for traj, logp in zip(batch, logps):
        traj["prox_logp"] = logp

    adv_batch = actor.compute_advantages(batch)
    actor.ppo_update(adv_batch)
    actor.step_lr_scheduler()

    rollout.pause()
    await asyncio.to_thread(actor.update_weights, weight_update_meta)
    rollout.resume()

    actor.set_version(global_step + 1)
    rollout.set_version(global_step + 1)

    step_time = time.perf_counter() - tik
    if global_step >= warmup_steps:
        times.append(step_time)

    log_data = {
        "train/global_step": global_step,
        "train/step_time_sec": step_time,
        "train/lr": actor.lr_scheduler.get_last_lr()[0],
        **metrics,
    }
    try:
        areal_stats = stats_tracker.export_all(reset=True)
        log_data.update({f"areal/{k}": v for k, v in areal_stats.items() if v is not None})
    except Exception as exc:
        log_data["areal/stats_export_failed"] = 1
        print(f"[wandb] stats export failed at step {global_step}: {exc}")
        reset_areal_stats_tracker()

    wandb.log(log_data, step=global_step)
    print(
        f"step={global_step:03d} "
        f"reward_mean={metrics['rollout/reward_mean']:.3f} "
        f"success={metrics['rollout/success_rate']:.2f} "
        f"seq_len={metrics['rollout/seq_len_mean']:.1f} "
        f"time={step_time:.2f}s"
    )

wandb_run.finish()
print(times)


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/ubuntu/.netrc.
wandb: Currently logged in as: zh1720 (julie3399) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, mcp, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


(AReaL) 20260524-22:45:54.325 PPOActor INFO: ======================================================================
(AReaL) 20260524-22:45:54.325 PPOActor INFO: PPOActor Configuration
(AReaL) 20260524-22:45:54.325 PPOActor INFO: ======================================================================
(AReaL) 20260524-22:45:54.326 PPOActor INFO: Mode: Decoupled PPO (off-policy)
(AReaL) 20260524-22:45:54.326 PPOActor INFO:   log_p_behave (π_behave): FROM INFERENCE (behavior policy)
(AReaL) 20260524-22:45:54.326 PPOActor INFO:   Proximal policy (π_prox): RECOMPUTED via forward pass (standard decoupled PPO)
(AReaL) 20260524-22:45:54.327 PPOActor INFO:   log_p_theta (π_θ): TRAINING FORWARD PASS (current policy)
(AReaL) 20260524-22:45:54.327 PPOActor INFO:   Rejection sampling: level=token, metric=ratio, action=mask, upper=5.0
(AReaL) 20260524-22:45:54.327 PPOActor INFO: ======================================================================
(AReaL) 20260524-22:45:54.328 PPOActor INFO: Training

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1767.50it/s]


(AReaL) 20260524-22:45:55.236 IOStruct INFO: Memory-Usage after model creation/loading: memory allocated (GB): 0.92, memory reserved (GB): 0.93, device memory used/total (GB): 23.66/44.39
(AReaL) 20260524-22:45:55.237 [FSDPEngine Rank 0] INFO: Model creation and loading time: 0.56s
(AReaL) 20260524-22:45:55.326 [FSDPEngine Rank 0] INFO: Applying FSDP2 with N-D parallelism for 0.09 seconds
(AReaL) 20260524-22:45:55.327 [FSDPEngine Rank 0] INFO: Create optimizer time: 0.0007604060001540347
(AReaL) 20260524-22:45:56.331 [RemoteInfEngine Rank 0] INFO: Failed to get server addresses from name_resolve, falling back to environment variable.
(AReaL) 20260524-22:45:56.332 [RemoteInfEngine Rank 0] INFO: Get server addresses from environment variable.
(AReaL) 20260524-22:45:56.332 [RemoteInfEngine Rank 0] INFO: Waiting for server ready...
(AReaL) 20260524-22:45:57.337 [RemoteInfEngine Rank 0] INFO: Servers are all ready!
(AReaL) 20260524-22:46:04.569 [RemoteInfEngine Rank 0] WARNING: Trajectory m

...

(AReaL) 20260524-22:55:56.052 [RemoteInfEngine Rank 0] INFO: Loading weights from disk done in 3.52s. Respond time: 0.91s.
step=048 reward_mean=0.607 success=0.62 seq_len=403.8 time=10.69s
(AReaL) 20260524-22:56:00.595 [RemoteInfEngine Rank 0] WARNING: Trajectory missing 'versions' field, defaulting to current inference engine version.
(AReaL) 20260524-22:56:03.153 [RemoteInfEngine Rank 0] WARNING: Trajectory missing 'versions' field, defaulting to current inference engine version.
(AReaL) 20260524-22:56:03.172 [FSDPEngine Rank 0] INFO: Microbatch #tokens (rank 0): [3799], padded to: [3840], padding lengths: [41]
(AReaL) 20260524-22:56:03.332 [FSDPEngine Rank 0] INFO: Microbatch #tokens (rank 0): [3799], padded to: [3840], padding lengths: [41]


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.62s/it]


(AReaL) 20260524-22:56:07.096 [RemoteInfEngine Rank 0] INFO: Loading weights from disk done in 3.51s. Respond time: 0.95s.
step=049 reward_mean=0.562 success=0.62 seq_len=474.9 time=11.04s
(AReaL) 20260524-22:56:07.672 [RemoteInfEngine Rank 0] WARNING: Trajectory missing 'versions' field, defaulting to current inference engine version.
(AReaL) 20260524-22:56:09.940 [RemoteInfEngine Rank 0] WARNING: Trajectory missing 'versions' field, defaulting to current inference engine version.
(AReaL) 20260524-22:56:09.960 [FSDPEngine Rank 0] INFO: Microbatch #tokens (rank 0): [2248], padded to: [2304], padding lengths: [56]
(AReaL) 20260524-22:56:10.060 [FSDPEngine Rank 0] INFO: Microbatch #tokens (rank 0): [2248], padded to: [2304], padding lengths: [56]


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.67s/it]


(AReaL) 20260524-22:56:13.787 [RemoteInfEngine Rank 0] INFO: Loading weights from disk done in 3.52s. Respond time: 0.91s.
step=050 reward_mean=0.533 success=0.75 seq_len=281.0 time=6.68s
